In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import random

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
from torch.optim import Adam
from torch.utils.data import Subset

In [ ]:
def set_seed(seed=100):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed()

In [ ]:
print("CUDA available:", torch.cuda.is_available())
print("Device:", "cuda" if torch.cuda.is_available() else "cpu")
device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
NPZ_PATH   = "dataset.npz"    # path to your .npz dataset
MODEL_PATH = "unet_best.pt"   # where the best checkpoint will be saved


In [ ]:
class InsoleNPZDataset(Dataset):
    """
    Loads X, Y from a NPZ file.
    X, Y are expected to be float arrays in [0,1] with -1 for invalid zones.
    Shapes: (N, 1, 33, 15)
    We build a 2-channel input: [X_filled, mask]
    """
    def __init__(self, npz_path: str):
        data = np.load(npz_path)
        self.X = data["X"].astype(np.float32)
        self.Y = data["Y"].astype(np.float32)

        assert self.X.ndim == 4 and self.Y.ndim == 4
        assert self.X.shape == self.Y.shape

    def __len__(self):
        return self.X.shape[0]

    def __getitem__(self, idx):
        x = self.X[idx]  # (1, 33, 15)
        y = self.Y[idx]  # (1, 33, 15)

        # valid where y != -1 (and usually x != -1 too)
        valid = (y != -1.0).astype(np.float32)  # (1,33,15) mask

        # fill invalid zones with 0 for network input / target
        x_filled = np.where(x != -1.0, x, 0.0).astype(np.float32)
        y_filled = np.where(y != -1.0, y, 0.0).astype(np.float32)

        # add mask as second channel
        x2 = np.concatenate([x_filled, valid], axis=0)  # (2,33,15)

        return torch.from_numpy(x2), torch.from_numpy(y_filled), torch.from_numpy(valid)

In [ ]:
def make_loaders(npz_path, batch_size=64, train_ratio=0.7, val_ratio=0.2):
    ds = InsoleNPZDataset(npz_path)

    N = len(ds)
    idx = list(range(N))

    train_idx = idx[:int(train_ratio*N)]
    val_idx   = idx[int(train_ratio*N):int((train_ratio+val_ratio)*N)]
    test_idx  = idx[int((train_ratio+val_ratio)*N):]

    train_ds = Subset(ds, train_idx)
    val_ds   = Subset(ds, val_idx)
    test_ds  = Subset(ds, test_idx)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, drop_last=False)
    val_loader   = DataLoader(val_ds, batch_size=batch_size, shuffle=False, drop_last=False)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, drop_last=False)

    print("Total dataset length: ", N)
    print("Train dataset length: ", len(train_ds))
    print("Validation dataset length: ", len(val_ds))
    print("Test dataset length: ", len(test_ds))

    return train_loader, val_loader, test_loader

In [ ]:
class DoubleConv(nn.Module):
    """
    Two consecutive Conv2D + BatchNorm + ReLU blocks
    """
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.net(x)

In [ ]:
class UNetSmall(nn.Module):
    """
    U-Net for small 2D insole images (33x15)

    Input  : (B, 2, 33, 15)
    Output : (B, 1, 33, 15)
    """

    def __init__(self, in_ch=2, out_ch=1, base=32):
        super().__init__()

        # Encoder
        self.enc1 = DoubleConv(in_ch, base)
        self.pool1 = nn.MaxPool2d(2)

        self.enc2 = DoubleConv(base, base * 2)
        self.pool2 = nn.MaxPool2d(2)

        # Bottleneck
        self.bottleneck = DoubleConv(base * 2, base * 4)

        # Decoder
        self.up2 = nn.ConvTranspose2d(base * 4, base * 2, kernel_size=2, stride=2)
        self.dec2 = DoubleConv(base * 4, base * 2)

        self.up1 = nn.ConvTranspose2d(base * 2, base, kernel_size=2, stride=2)
        self.dec1 = DoubleConv(base * 2, base)

        # Output
        self.out = nn.Conv2d(base, out_ch, kernel_size=1)

    def forward(self, x):
        e1 = self.enc1(x)
        p1 = self.pool1(e1)

        e2 = self.enc2(p1)
        p2 = self.pool2(e2)

        b = self.bottleneck(p2)

        u2 = self.up2(b)
        u2 = self._align(u2, e2)
        d2 = self.dec2(torch.cat([u2, e2], dim=1))

        u1 = self.up1(d2)
        u1 = self._align(u1, e1)
        d1 = self.dec1(torch.cat([u1, e1], dim=1))

        y = self.out(d1)
        return torch.sigmoid(y)

    @staticmethod
    def _align(src, ref):
        """
        Pad or crop src to match spatial size of ref
        """
        _, _, h, w = ref.shape
        src = F.pad(src, (0, max(0, w - src.shape[-1]),
                          0, max(0, h - src.shape[-2])))
        return src[:, :, :h, :w]

In [ ]:
def masked_smooth_l1(pred, target, mask, beta=0.02):
    """
    Smooth L1 loss computed only on valid sensor pixels
    """
    loss_map = F.smooth_l1_loss(pred, target, reduction="none", beta=beta)
    loss_map = loss_map * mask
    return loss_map.sum() / mask.sum().clamp_min(1.0)

In [ ]:
def train_unet(npz_path, epochs=50, batch_size=64, lr=1e-3, patience=10, min_delta=5e-5, device=device):
    train_loader, val_loader, _ = make_loaders(npz_path, batch_size)

    model = UNetSmall().to(device)
    optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-3,
    weight_decay=1e-4
)

    train_losses = []
    val_losses = []

    best_val = float("inf")
    bad_epochs = 0

    for epoch in range(1, epochs + 1):
        # ---- Training ----
        model.train()
        train_loss = 0.0

        for x2, y, mask in train_loader:
            x2, y, mask = x2.to(device), y.to(device), mask.to(device)

            pred = model(x2)
            loss_map = (pred - y) ** 2
            loss_map = loss_map * mask
            loss = loss_map.sum() / mask.sum().clamp_min(1.0)


            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            train_loss += loss.item()

        train_loss /= len(train_loader)
        train_losses.append(train_loss)

        # ---- Validation ----
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for x2, y, mask in val_loader:
                x2, y, mask = x2.to(device), y.to(device), mask.to(device)
                pred = model(x2)
                val_loss += masked_smooth_l1(pred, y, mask).item()

        val_loss /= len(val_loader)
        val_losses.append(val_loss)

        print(f"Epoch {epoch:03d} | train={train_loss:.6f} | val={val_loss:.6f}")

        # ---- Save best + Early stopping ----
        if val_loss < best_val - min_delta:
            best_val = val_loss
            bad_epochs = 0
            torch.save(model.state_dict(), MODEL_PATH)
            print("  ↳ improved, saved best")
        else:
            bad_epochs += 1
            print(f"  ↳ no improvement ({bad_epochs}/{patience})")

            if bad_epochs >= patience:
                print("Early stopping triggered.")
                break

    print("Training finished. Best model saved as unet_best.pt")
    return model, train_losses, val_losses

In [ ]:
model, train_losses, val_losses = train_unet(
  npz_path=NPZ_PATH,
  epochs=12,
  batch_size=64,
  lr=1e-3
)

In [ ]:
model = UNetSmall().to(device)
model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
model.eval()

In [ ]:
_, _, test_loader = make_loaders(npz_path=NPZ_PATH, batch_size=64)

In [ ]:
def masked_mae_torch(a, b, mask):
    # a,b,mask: (B,1,33,15) ; mask is 1 valid, 0 ignore
    diff = (a - b).abs() * mask
    denom = mask.sum().clamp_min(1.0)
    return (diff.sum() / denom).item()

In [ ]:
def masked_rmse_torch(pred, target, mask, eps=1e-8):
    # pred, target: (B,1,33,15)
    # mask:         (B,1,33,15) or (B,33,15)
    if mask.dim() == 3:
        mask = mask.unsqueeze(1)

    diff2 = (pred - target) ** 2
    diff2 = diff2 * mask

    mse = diff2.sum() / (mask.sum() + eps)
    rmse = torch.sqrt(mse)
    return rmse

In [ ]:
def masked_ssim_torch(pred, target, mask, C1=0.01**2, C2=0.03**2):
    """
    pred, target: (B,1,H,W)
    mask:         (B,1,H,W) or (B,H,W)
    Returns: mean SSIM over batch
    """
    if mask.dim() == 3:
        mask = mask.unsqueeze(1)

    # Keep only valid pixels
    pred = pred * mask
    target = target * mask

    # Number of valid pixels per image
    N = mask.sum(dim=(2,3), keepdim=True).clamp(min=1)

    # Means
    mu_x = (pred.sum(dim=(2,3), keepdim=True)) / N
    mu_y = (target.sum(dim=(2,3), keepdim=True)) / N

    # Variances and covariance
    sigma_x = ((pred - mu_x)**2 * mask).sum(dim=(2,3), keepdim=True) / N
    sigma_y = ((target - mu_y)**2 * mask).sum(dim=(2,3), keepdim=True) / N
    sigma_xy = ((pred - mu_x)*(target - mu_y) * mask).sum(dim=(2,3), keepdim=True) / N

    # SSIM map (per image)
    ssim = ((2*mu_x*mu_y + C1)*(2*sigma_xy + C2)) / \
           ((mu_x**2 + mu_y**2 + C1)*(sigma_x + sigma_y + C2))

    return ssim.mean()

In [ ]:
def _to_float(x):
    return x.item() if hasattr(x, "item") else float(x)

@torch.no_grad()
def eval_on_loader(model, loader, device):
    model.eval()

    mae_ct_ref = 0.0
    mae_pred_ref = 0.0
    rmse_ct_ref = 0.0
    rmse_pred_ref = 0.0
    ssim_ct_ref = 0.0
    ssim_pred_ref = 0.0

    n_batches = 0

    for x2, y, mask in loader:
        x2, y, mask = x2.to(device), y.to(device), mask.to(device)

        ct = x2[:, 0:1, :, :]
        pred = model(x2)

        mae_ct_ref   += _to_float(masked_mae_torch(ct,   y, mask))
        mae_pred_ref += _to_float(masked_mae_torch(pred, y, mask))

        rmse_ct_ref   += _to_float(masked_rmse_torch(ct,   y, mask))
        rmse_pred_ref += _to_float(masked_rmse_torch(pred, y, mask))

        ssim_ct_ref   += masked_ssim_torch(ct,   y, mask)
        ssim_pred_ref += masked_ssim_torch(pred, y, mask)

        n_batches += 1

    mae_ct_ref   /= n_batches
    mae_pred_ref /= n_batches
    rmse_ct_ref   /= n_batches
    rmse_pred_ref /= n_batches
    ssim_ct_ref   /= n_batches
    ssim_pred_ref /= n_batches

    print("MAE  (CT   vs REF):", mae_ct_ref)
    print("MAE  (PRED vs REF):", mae_pred_ref)
    print("MAE Improvement:", mae_ct_ref - mae_pred_ref)

    print("-" * 60)

    print("RMSE (CT   vs REF):", rmse_ct_ref)
    print("RMSE (PRED vs REF):", rmse_pred_ref)
    print("RMSE Improvement:", rmse_ct_ref - rmse_pred_ref)

    print("-" * 60)

    print("SSIM (CT   vs REF):", float(ssim_ct_ref))
    print("SSIM (PRED vs REF):", float(ssim_pred_ref))
    print("SSIM Improvement:", float(ssim_pred_ref - ssim_ct_ref))


In [ ]:
# --- Load dataset ---
data = np.load(NPZ_PATH)
X = data["X"]  # CT  (N,1,33,15)
Y = data["Y"]  # REF (N,1,33,15)

# valid mask: 1 where sensor exists, 0 outside (-1)
mask = (Y != -1).astype(np.float32)

# fill outside with 0
Xf = np.where(mask == 1, X, 0.0)
Yf = np.where(mask == 1, Y, 0.0)

eval_on_loader(model=model, loader=test_loader, device=device)

In [ ]:
np_train_losses = np.array(train_losses)
np_val_losses = np.array(val_losses)

gap = np_val_losses - np_train_losses

min_gap_epoch = np.argmin(gap)
last_epoch = len(gap) - 1

fig, ax1 = plt.subplots(figsize=(17,12.5))

train_color = "#003B5C"
val_color = "green"
gap_color = "#2A9D8F"
text_color = "#0B4F5C"

ax1.plot(np_train_losses, label="Train Loss", linewidth=3, color=train_color)
ax1.plot(np_val_losses, label="Validation Loss", linewidth=3, color=val_color)

ax1.set_xlabel("Epoch", fontsize=35, labelpad=15, color=text_color)
ax1.set_ylabel("Loss", fontsize=35, labelpad=15, color=text_color)
ax1.tick_params(axis='both', labelsize=24, colors=text_color)
ax1.grid(True)

ax2 = ax1.twinx()
ax2.plot(gap, linestyle="--", color=gap_color, linewidth=3, label="Generalisation Gap")
ax2.set_ylabel("Gap (Val - Train)", fontsize=35, labelpad=15, color=text_color)
ax2.tick_params(axis='y', labelsize=24, colors=text_color)

ax1.axvline(min_gap_epoch, linestyle="--", color="red",
            linewidth=3, label="Best generalisation (min gap)")
ax1.axvline(last_epoch, linestyle="--", color="darkred",
            linewidth=3, label="Early stopping (validation loss)")

ax2.scatter(
    min_gap_epoch,
    gap[min_gap_epoch],
    color="red",
    marker="x",
    s=250,
    linewidths=4,
    zorder=6
)

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()

legend = ax1.legend(
    lines1 + lines2,
    labels1 + labels2,
    loc="upper center",
    bbox_to_anchor=(0.6, 1.02),
    fontsize=24
)

for text in legend.get_texts():
    text.set_color(text_color)

plt.title(
    "Learning Curves and Generalisation Gap",
    fontsize=50,
    pad=60,
    color=text_color
)

plt.tight_layout()
plt.subplots_adjust(top=0.88, bottom=0.12, left=0.12, right=0.88)

plt.show()

In [ ]:
overfitting_index = (val_losses[-1] - train_losses[-1]) / train_losses[-1]
print("Overfitting Index:", overfitting_index)